# Capstone EDA

This notebook performs exploratory data analysis for the capstone panel dataset. It follows the required structure and saves publication-quality figures to `results/figures/` using `FIGURES_DIR` from `config_paths`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

sys.path.append(str(Path.cwd() / 'code'))
from config_paths import FINAL_DATA_DIR, FIGURES_DIR

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 14
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

data_path = FINAL_DATA_DIR / 'crypto_analysis_panel.csv'
df = pd.read_csv(data_path, parse_dates=['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

# Compute the outcome variable as monthly log return by symbol
df['log_price'] = np.log(df['price'])
df['outcome'] = df.groupby('symbol')['log_price'].diff()
df = df.dropna(subset=['outcome']).reset_index(drop=True)

# Derived variables for analysis
df['market_cap_billion'] = df['market_cap'] / 1e9

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

## Section 2: Summary Statistics
This section summarizes the main variables and checks basic data quality.

In [ ]:
display(df.head())

summary_stats = df[['outcome', 'fed_funds_rate', 'vix', 'epu_index', 'market_cap_billion']].describe()
missing_values = df.isna().sum()
symbol_counts = df['symbol'].value_counts()

summary_stats

print('Missing values by column:')
print(missing_values)

print('Observations per symbol:')
print(symbol_counts)

## Section 3: Correlation Analysis
This section explores correlations between the outcome, main driver, and control variables.

In [ ]:
vars_to_plot = ['outcome', 'fed_funds_rate', 'vix', 'epu_index', 'market_cap_billion']
corr_matrix = df[vars_to_plot].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap=sns.color_palette('coolwarm', as_cmap=True), center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap: Outcome, Driver, and Controls', fontsize=16)
plt.tight_layout()
output_path = FIGURES_DIR / 'correlation_heatmap.png'
plt.savefig(output_path, dpi=300)
plt.show()
print('Caption: The heatmap reveals the strongest relationships between the outcome and macro controls, highlighting potential multicollinearity risks.')

corr_matrix

## Section 4: Time Series
This section visualizes the aggregate outcome time series across the panel.

In [ ]:
outcome_ts = df.groupby('date')['outcome'].mean().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(outcome_ts.index, outcome_ts.values, color='tab:blue', linewidth=2)
plt.title('Average Monthly Log Return of Crypto Prices', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Average Log Return', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
output_path = FIGURES_DIR / 'time_series_outcome.png'
plt.savefig(output_path, dpi=300)
plt.show()

print('Caption: The aggregate outcome series shows periods of higher volatility and persistent movement that likely reflect macroeconomic shocks and market sentiment shifts.')

In [ ]:
outcome_ts = df.groupby('date')['outcome'].mean().sort_index()
driver_ts = df.groupby('date')['fed_funds_rate'].mean().sort_index()

fig, ax1 = plt.subplots(figsize=(12, 6))
color_outcome = 'tab:blue'
color_driver = 'tab:green'
ax1.plot(outcome_ts.index, outcome_ts.values, color=color_outcome, label='Average outcome', linewidth=2)
ax1.set_xlabel('Date', fontsize=14)
ax1.set_ylabel('Average Log Return', color=color_outcome, fontsize=14)
ax1.tick_params(axis='y', labelcolor=color_outcome)

ax2 = ax1.twinx()
ax2.plot(driver_ts.index, driver_ts.values, color=color_driver, label='Average Fed Funds Rate', linewidth=2, linestyle='--')
ax2.set_ylabel('Fed Funds Rate (%)', color=color_driver, fontsize=14)
ax2.tick_params(axis='y', labelcolor=color_driver)

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper left', fontsize=12)
plt.title('Average Outcome and Fed Funds Rate Over Time', fontsize=16)
ax1.grid(True, linestyle='--', alpha=0.4)
fig.tight_layout()
output_path = FIGURES_DIR / 'dual_axis_outcome_driver.png'
fig.savefig(output_path, dpi=300)
plt.show()
print('Caption: The dual-axis plot shows how the outcome and interest rate move relative to one another and whether the rate leads or lags the average crypto return.')


## Section 5: Lagged Effect Analysis
This section identifies the lag at which the driver variable has the strongest correlation with the outcome.

In [ ]:
lags = [0, 1, 2, 3, 6, 12]  # months
correlations = {}

for lag in lags:
    df[f'fed_funds_rate_lag_{lag}'] = df.groupby('symbol')['fed_funds_rate'].shift(lag)
    valid = df[['outcome', f'fed_funds_rate_lag_{lag}']].dropna()
    correlations[lag] = valid['outcome'].corr(valid[f'fed_funds_rate_lag_{lag}'])

corr_series = pd.Series(correlations)
optimal_lag = corr_series.abs().idxmax()

plt.figure(figsize=(10, 5))
sns.barplot(x=corr_series.index.astype(str), y=corr_series.values, palette='colorblind')
plt.axhline(0, color='black', linewidth=1)
plt.title('Correlation Between Outcome and Lagged Fed Funds Rate', fontsize=16)
plt.xlabel('Lag (months)', fontsize=14)
plt.ylabel('Pearson Correlation', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
output_path = FIGURES_DIR / 'lagged_correlation_bar.png'
plt.savefig(output_path, dpi=300)
plt.show()
print('Caption: The lag chart identifies which lag of the fed funds rate has the strongest association with the outcome, informing the M3 specification.')

print(f'Optimal lag: {optimal_lag} months with correlation = {corr_series.loc[optimal_lag]:.3f}')

## Section 6: Group Analysis
This section compares outcomes across crypto symbols and measures symbol-level sensitivity.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='symbol', y='outcome', palette='colorblind', linewidth=1.2)
plt.title('Distribution of Monthly Log Returns by Crypto Symbol', fontsize=16)
plt.xlabel('Crypto Symbol', fontsize=14)
plt.ylabel('Monthly Log Return', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.4)
sns.despine(trim=True)
plt.tight_layout()
output_path = FIGURES_DIR / 'group_boxplot_outcome.png'
plt.savefig(output_path, dpi=300)
plt.show()
print('Caption: The boxplot compares return distributions across crypto symbols and highlights dispersion and median differences.')

symbol_corr = df.groupby('symbol').apply(lambda group: group['outcome'].corr(group['fed_funds_rate'])).sort_values()
threshold = -0.1
colors = ['red' if val < threshold else 'steelblue' for val in symbol_corr] 

plt.figure(figsize=(10, 6))
plt.barh(symbol_corr.index.astype(str), symbol_corr.values, color=colors)
plt.axvline(threshold, color='gray', linestyle='--', linewidth=1)
plt.title('Symbol-Level Sensitivity to Fed Funds Rate', fontsize=16)
plt.xlabel('Correlation with Outcome', fontsize=14)
plt.ylabel('Crypto Symbol', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
output_path = FIGURES_DIR / 'group_sensitivity_bar.png'
plt.savefig(output_path, dpi=300)
plt.show()
print('Caption: The symbol-level sensitivity bar chart flags which assets are most responsive to the fed funds rate.')


## Section 7: Factor / Control Relationships
This section visualizes the relationship between the outcome and control variables.

In [ ]:
control_vars = ['vix', 'epu_index']
for control in control_vars:
    plt.figure(figsize=(10, 6))
    sns.regplot(data=df, x=control, y='outcome', scatter_kws={'alpha': 0.4, 's': 40}, line_kws={'color': 'firebrick', 'linewidth': 2}, ci=95)
    plt.title(f'Outcome vs {control.replace('_', ' ').title()}', fontsize=16)
    plt.xlabel(f'{control.replace('_', ' ').title()} (units)', fontsize=14)
    plt.ylabel('Monthly Log Return', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    output_path = FIGURES_DIR / f'outcome_vs_{control}.png'
    plt.savefig(output_path, dpi=300)
    plt.show()
    print('Caption: The scatter plot with regression line shows how the control variable explains variation in the outcome.')

## Section 8: Time Series Decomposition
This section decomposes the aggregate outcome series into observed, trend, seasonal, and residual components.

In [ ]:
series = df.groupby('date')['outcome'].mean().sort_index()
decomp = seasonal_decompose(series, model='additive', period=12, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
axes[0].plot(decomp.observed, color='black')
axes[0].set_title('Observed Outcome Series', fontsize=14)
axes[0].set_ylabel('Outcome', fontsize=12)

axes[1].plot(decomp.trend, color='tab:blue')
axes[1].set_title('Trend Component', fontsize=14)
axes[1].set_ylabel('Trend', fontsize=12)

axes[2].plot(decomp.seasonal, color='tab:green')
axes[2].set_title('Seasonal Component', fontsize=14)
axes[2].set_ylabel('Seasonal', fontsize=12)

axes[3].plot(decomp.resid, color='tab:red')
axes[3].set_title('Residual Component', fontsize=14)
axes[3].set_ylabel('Residual', fontsize=12)
axes[3].set_xlabel('Date', fontsize=12)

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
output_path = FIGURES_DIR / 'time_series_decomposition.png'
fig.savefig(output_path, dpi=300)
plt.show()
print('Caption: The decomposition separates observed, trend, seasonal, and residual components to guide time series modeling.')